In [ ]:
# ==============================================================================
# ⚙️ SYSTEM CONFIGURATION
# Modify these variables to adapt the script to your specific solar installation.
# ==============================================================================

# 1. Geographic Location
LATITUDE = 10.9685  # e.g., Barranquilla Latitude
LONGITUDE = -74.7813 # e.g., Barranquilla Longitude
CITY_NAME = "Barranquilla"
COUNTRY_NAME = "Colombia"
TIMEZONE_STR = "America/Bogota"

# 2. Forecasting Target
# Set to None to automatically forecast 'Tomorrow' based on your dataset
# Set to a specific date string (e.g., '2026-04-14') to force a simulation for that date
FORECAST_DATE_OVERRIDE = None

# 2. System Hardware Constants
SYSTEM_MAX_CAPACITY_W = 8700     # Maximum theoretical output of your inverter(s) in Watts

# 3. Data Extraction
DATA_PATH = './Data/'
RESULTS_PATH = './Results/'
# Define exactly which columns in your Excel files represent Solar Power (DC or AC). 
# The script will sum these columns to find the total power generated.
PV_COLUMNS = ['ppv1', 'ppv2', 'ppv3']  

# ==============================================================================
# 🚀 CORE PROCESSING ENGINE (Do not modify below this line)
# ==============================================================================
import os
import glob
import pandas as pd
import numpy as np
import datetime
import pytz
import matplotlib.pyplot as plt
from astral import LocationInfo
from astral.sun import sun
import warnings
warnings.filterwarnings('ignore')

# 1. Configure Location Strategy
city = LocationInfo(CITY_NAME, COUNTRY_NAME, TIMEZONE_STR, LATITUDE, LONGITUDE)
local_tz = pytz.timezone(TIMEZONE_STR)

# 2. Find and Load Local Data (Excel Format)
excel_files = glob.glob(os.path.join(DATA_PATH, '*.xls*'))
if not excel_files:
    raise ValueError(f"No Excel files found in directory: {DATA_PATH}")

df_list = []
for file in excel_files:
    try:
        temp_df = pd.read_excel(file, skiprows=2)
        df_list.append(temp_df)
    except Exception as e:
        print(f"Error reading {file}: {e}")

df = pd.concat(df_list, ignore_index=True)

# 3. Time Parsing
df['Time'] = pd.to_datetime(df['Time'])
df['date'] = df['Time'].dt.date
df['time'] = df['Time'].dt.time
df = df.sort_values(by='Time')

# 4. Filter Sunrise/Sunset using Astral
df_sun = pd.DataFrame()
for d in df['date'].unique():
    try:
        s = sun(city.observer, date=d)
        sunrise = s['sunrise'].astimezone(local_tz).time()
        sunset = s['sunset'].astimezone(local_tz).time()
        day_data = df[df['date'] == d].copy()
        day_data = day_data[(day_data['time'] >= sunrise) & (day_data['time'] <= sunset)]
        df_sun = pd.concat([df_sun, day_data])
    except Exception:
        pass
df = df_sun

# 5. Extract Solar Power Generation
# Ensure columns exist, padding missing ones with 0 to prevent errors
for col in PV_COLUMNS:
    if col not in df.columns:
        df[col] = 0

df['Solar_Power_W'] = df[PV_COLUMNS].fillna(0).sum(axis=1)

# Ensure precise 15-min intervals
df = df.drop_duplicates(subset=['Time'])
df = df.set_index('Time')
df_total_15min = df[['Solar_Power_W']].resample('15min').mean()

# 6. Time Range Discovery
earliest_day = df_total_15min.index.min().date()
latest_day = df_total_15min.index.max().date()
start_date_str = earliest_day.strftime("%Y-%m-%d")
end_date_str = latest_day.strftime("%Y-%m-%d")

# Fetch Today's real curve
df_today = df_total_15min[df_total_15min.index.date == latest_day].copy()

# 7. Fetch Open-Meteo Historical GHI
import openmeteo_requests
import requests_cache
from retry_requests import retry

def get_ghi_openmeteo_historical(lat, lon, start_str, end_str):
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_str,
        "end_date": end_str,
        "hourly": "shortwave_radiation",
        "timezone": TIMEZONE_STR
    }
    
    responses = openmeteo.weather_api(url, params=params)
    hourly = responses[0].Hourly()
    ghi = hourly.Variables(0).ValuesAsNumpy()
    hourly_data = {"ds": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True).tz_convert(TIMEZONE_STR),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True).tz_convert(TIMEZONE_STR),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )}
    hourly_data["GHI"] = ghi
    df_ghi_hourly = pd.DataFrame(data=hourly_data).set_index('ds')
    df_ghi = df_ghi_hourly.resample('15min').interpolate(method='linear').reset_index()
    return df_ghi

df_ghi_15min = get_ghi_openmeteo_historical(LATITUDE, LONGITUDE, start_date_str, end_date_str)
print("Data processing complete. Proceed to execution cell.")


In [ ]:
# ==============================================================================
# ☀️ FORECAST EXECUTION (Direct Physical Scaling Engine)
# ==============================================================================

# 1. Establish Proven System Capacity (Auto-scaling based on last 7 days of hardware output)
MAX_POWER_HISTORY = df_total_15min['Solar_Power_W'].tail(7 * 96).max()

# 2. Establish Maximum Historical Irradiance
MAX_GHI_HISTORY = df_ghi_15min['GHI'].max()
if MAX_GHI_HISTORY == 0 or pd.isna(MAX_GHI_HISTORY):
    MAX_GHI_HISTORY = 1000 

THERMAL_EFFICIENCY_FACTOR = MAX_POWER_HISTORY / MAX_GHI_HISTORY

print(f"Detected Hardware Peak (Last 7 days): {MAX_POWER_HISTORY:.2f} W")
print(f"Detected Ghi Ceiling: {MAX_GHI_HISTORY:.2f} W/m2")
print(f"Calculated Efficiency Rating: {THERMAL_EFFICIENCY_FACTOR:.2f} (W generated per W/m2)\n")

# Set forecast target to tomorrow relative to the dataset
if FORECAST_DATE_OVERRIDE:
    forecast_date = datetime.datetime.strptime(FORECAST_DATE_OVERRIDE, '%Y-%m-%d').date()
else:
    forecast_date = latest_day + datetime.timedelta(days=1)
forecast_date_str = forecast_date.strftime("%Y-%m-%d")

future = pd.date_range(
    start=datetime.datetime.combine(forecast_date, datetime.time(0,0)),
    end=datetime.datetime.combine(forecast_date, datetime.time(23,45)),
    freq='15min'
).to_frame(index=False, name='ds')

# 3. Fetch Forecast Data (Including Clear Sky envelope)
def get_ghi_and_clearsky_forecast(lat, lon, date_str):
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon, "start_date": date_str, "end_date": date_str,
        "hourly": ["shortwave_radiation", "shortwave_radiation_clear_sky"],
        "timezone": TIMEZONE_STR
    }
    try:
        responses = openmeteo.weather_api(url, params=params)
    except Exception:
        # Fallback to forecast API if date is in the actual future
        url = "https://api.open-meteo.com/v1/forecast"
        params["hourly"] = ["shortwave_radiation", "shortwave_radiation_clear_sky"] # clear sky might not be perfect in forecast, but we try
        responses = openmeteo.weather_api(url, params=params)

    hourly = responses[0].Hourly()
    ghi = hourly.Variables(0).ValuesAsNumpy()
    clear_sky = hourly.Variables(1).ValuesAsNumpy()
    
    data = {"ds": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True).tz_convert(TIMEZONE_STR).tz_localize(None),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True).tz_convert(TIMEZONE_STR).tz_localize(None),
        freq=pd.Timedelta(seconds=hourly.Interval()), inclusive="left"
    )}
    data["GHI"] = ghi
    data["ClearSky"] = clear_sky
    
    df = pd.DataFrame(data).set_index('ds')
    df_15min = df.resample('15min').interpolate(method='linear').reset_index()
    df_15min['GHI'] = df_15min['GHI'].apply(lambda x: 0 if pd.isna(x) or x < 0 else x)
    df_15min['ClearSky'] = df_15min['ClearSky'].apply(lambda x: 0 if pd.isna(x) or x < 0 else x)
    return df_15min

df_ghi_future = get_ghi_and_clearsky_forecast(LATITUDE, LONGITUDE, forecast_date_str)

# 4. Core Prediction Calculations
future = pd.merge(future, df_ghi_future, on='ds', how='left')
future['GHI'] = future['GHI'].fillna(0)
future['ClearSky'] = future['ClearSky'].fillna(0)

future['yhat'] = future['GHI'] * THERMAL_EFFICIENCY_FACTOR
future['yhat_clearsky'] = future['ClearSky'] * THERMAL_EFFICIENCY_FACTOR

# Native Sunrise/Sunset Trim
future['date'] = future['ds'].dt.date
future['time'] = future['ds']
future['sunrise'] = pd.NaT
future['sunset']  = pd.NaT

for d in future['date'].unique():
    s = sun(city.observer, date=d)
    sunrise_local = s['sunrise'].astimezone(local_tz).replace(tzinfo=None)
    sunset_local = s['sunset'].astimezone(local_tz).replace(tzinfo=None)
    future.loc[future['date'] == d, 'sunrise'] = sunrise_local
    future.loc[future['date'] == d, 'sunset']  = sunset_local

future = future[(future['time'] >= future['sunrise']) & (future['time'] <= future['sunset'])]
future = future.drop(columns=['date','time','sunrise','sunset'])

# Apply hardware ceiling capacity bounds
future['yhat'] = future['yhat'].clip(lower=0, upper=SYSTEM_MAX_CAPACITY_W)
future['yhat_clearsky'] = future['yhat_clearsky'].clip(lower=0, upper=SYSTEM_MAX_CAPACITY_W)

# Variability Bounds
future['yhat_lower'] = (future['yhat'] * 0.90).clip(lower=0)
future['yhat_upper'] = (future['yhat'] * 1.10).clip(upper=SYSTEM_MAX_CAPACITY_W)

# Prepare DataFrame and Export Target
forecast_next = future.copy()
forecast_next['Hour_decimal'] = forecast_next['ds'].dt.hour + forecast_next['ds'].dt.minute/60

os.makedirs(RESULTS_PATH, exist_ok=True)
forecast_out = forecast_next[['ds', 'yhat', 'yhat_clearsky', 'yhat_lower', 'yhat_upper']]
forecast_out_path = os.path.join(RESULTS_PATH, 'forecast_hourly_table.csv')
forecast_out.to_csv(forecast_out_path, index=False)

# 5. Visual Output
plt.figure(figsize=(12,6))
if not df_today.empty:
    plt.plot(df_today.index.hour + df_today.index.minute/60, df_today['Solar_Power_W'].values, marker='o', linestyle='-', color='orange', label='Latest Daily Yield (Active System)')

plt.plot(forecast_next['Hour_decimal'], forecast_next['yhat_clearsky'], linestyle='--', color='gray', alpha=0.6, label='Installed Capacity Potential (Clear Sky Envelope)')
plt.plot(forecast_next['Hour_decimal'], forecast_next['yhat'], marker='o', linestyle='-', color='blue', label=f'Weather-Adjusted Forecast ({forecast_date_str})')
plt.fill_between(forecast_next['Hour_decimal'], forecast_next['yhat_lower'], forecast_next['yhat_upper'], color='lightblue', alpha=0.5)

plt.title('Autonomous Solar Yield Forecast')
plt.xlabel('Hour of Day')
plt.ylabel('Solar Output Power (W)')
plt.xticks(np.arange(0,24,1))
plt.grid(True)
plt.legend()
plt.tight_layout()

forecast_img_path = os.path.join(RESULTS_PATH, 'forecast_yield_curve.png')
plt.savefig(forecast_img_path)
plt.close()
